In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("/content/rfm_anomaly_scored.csv")
clv_df = pd.read_csv("/content/clv_scored.csv")

In [4]:
df = df.merge(
    clv_df[["CustomerID", "clv_12months", "clv_segment",
            "prob_alive", "predicted_purchases_90d"]],
    on = "CustomerID",
    how = "left"
)

In [6]:
def build_dataset_summary(df):
  total_customers = len(df)
  total_revenue = df["Monetary"].sum()
  avg_clv = df["clv_12months"].mean()
  total_anomalies = df["is_anomaly"].sum()

  segment_summary = df.groupby("Segment").agg(
      count = ("CustomerID", "count"),
      avg_recency = ("Recency", "mean"),
      avg_frequency = ("Frequency", "mean"),
      avg_monetary = ("Monetary", "mean"),
      avg_clv = ("clv_12months", "mean"),
      avg_prob_alive = ("prob_alive", "mean")
  ).round(2)

  anomaly_summary = df[df["is_anomaly"] == 1]["anomaly_type"].value_counts()

  clv_summary = df.groupby("clv_segment")["clv_12months"].agg(
      ["count", "mean"]
  ).round(2)

  return {
      "total_customer": total_customers,
      "total_revenue": round(total_revenue, 2),
      "avg_clv": round(avg_clv, 2),
      "total_anomalies": int(total_anomalies),
      "segment_summary": segment_summary.to_dict(),
      "anomaly_summary": anomaly_summary.to_dict(),
      "clv_summary": clv_summary.to_dict()
  }

In [7]:
summary = build_dataset_summary(df)

In [9]:
%%capture
!pip install groq

In [10]:
from groq import Groq

In [ ]:
client = Groq(api_key="")

In [29]:
MODEL = "qwen/qwen3-32b"

In [30]:
import json

In [58]:
def generate_executive_summary(summary: dict) -> str:
  prompt = f"""
  You are a senior customer analytics consultant presenting findings to a business executive.
  Analyze this customer data and provide a concise executive summary.

  Dataset Overview:
  - Total Customers: {summary['total_customer']}
  - Total Historical Revenue: £{summary['total_revenue']:,.2f}
  - Average Predicted CLV (12 months): £{summary['avg_clv']:,.2f}
  - Anomalous Customers Detected: {summary['total_anomalies']}

  Segment Distribution:
  {json.dumps(summary['segment_summary']['count'], indent=2)}

  Average Monetary Value per Segment:
  {json.dumps(summary['segment_summary']['avg_monetary'], indent=2)}

  Average CLV per Segment:
  {json.dumps(summary['segment_summary']['avg_clv'], indent=2)}

  CLV Tier Summary:
  {json.dumps(summary['clv_summary'], indent=2)}

  Write a 3-4 paragraph executive summary that:
  1. Highlights the most important findings
  2. Identifies the biggest opportunities
  3. Flags the most urgent risks
  4. Keeps language business-friendly, not technical
  Do not use bullet points. Write in clear business prose.
  Do NOT explain your thought process. Give only the summary.
  """

  response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=600
    )
  return response.choices[0].message.content

In [59]:
def generate_segment_recommendations(summary: dict) -> dict:
    prompt = f"""
You are a CRM and customer retention strategist.
Generate specific, actionable recommendations for each customer segment.

Segment Data (avg values):
Recency (days since last purchase):
{json.dumps(summary['segment_summary']['avg_recency'], indent=2)}

Frequency (number of purchases):
{json.dumps(summary['segment_summary']['avg_frequency'], indent=2)}

Monetary (total spend £):
{json.dumps(summary['segment_summary']['avg_monetary'], indent=2)}

Probability Still Active:
{json.dumps(summary['segment_summary']['avg_prob_alive'], indent=2)}

For each segment provide exactly:
1. A one-line description of who these customers are
2. The single most important action to take
3. A specific campaign or offer idea
4. Priority level (Critical / High / Medium / Low)

You MUST respond ONLY with a valid JSON object.
No explanation, no markdown, no code fences.
Start your response with {{ and end with }}

Example format:
{{
  "Champion": {{
    "description": "Best customers who buy often and recently",
    "action": "Reward loyalty and encourage referrals",
    "campaign": "VIP early access to new products",
    "priority": "High"
  }}
}}
"""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are a JSON-only response bot. You never output anything except valid JSON. No markdown, no explanation, no code fences."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.1,
        max_tokens=1500
    )

    raw = response.choices[0].message.content.strip()

    print("Raw model response:")
    print(raw)
    print("---")

    if "```json" in raw:
        raw = raw.split("```json")[1].split("```")[0].strip()
    elif "```" in raw:
        raw = raw.split("```")[1].split("```")[0].strip()

    if not raw.startswith("{"):
        start = raw.find("{")
        end = raw.rfind("}") + 1
        if start != -1 and end > start:
            raw = raw[start:end]

    if not raw:
        print("Model returned empty response — using fallback")
        return get_fallback_recommendations()

    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        print(f"JSON parse failed: {e}")
        print(f"Attempted to parse: {raw[:200]}")
        return get_fallback_recommendations()


def get_fallback_recommendations() -> dict:
    """Rule-based fallback if LLM fails"""
    return {
        "Champion": {
            "description": "Best customers — frequent, recent, high spend",
            "action": "Retain and reward loyalty",
            "campaign": "VIP program with early access and exclusive offers",
            "priority": "High"
        },
        "At Risk": {
            "description": "Previously good customers going quiet",
            "action": "Immediate win-back campaign",
            "campaign": "Personalised re-engagement email with discount",
            "priority": "Critical"
        },
        "Lost Customers": {
            "description": "Inactive customers with low recent engagement",
            "action": "Last chance reactivation or write off",
            "campaign": "One-time steep discount with urgency deadline",
            "priority": "Medium"
        },
        "Loyal Customer": {
            "description": "Consistent buyers with solid spend history",
            "action": "Upsell to higher value products",
            "campaign": "Exclusive member pricing on premium products",
            "priority": "High"
        },
        "New Customer": {
            "description": "Recent first-time buyers",
            "action": "Drive second purchase quickly",
            "campaign": "Welcome series with second purchase incentive",
            "priority": "High"
        },
        "Promising": {
            "description": "Recent high first order, not yet repeat",
            "action": "Fast-track to loyalty program",
            "campaign": "Personal follow-up within 7 days of first order",
            "priority": "High"
        },
        "Potential Loyalist": {
            "description": "Showing signs of becoming loyal",
            "action": "Nurture toward loyalty",
            "campaign": "Points-based rewards for next 3 purchases",
            "priority": "Medium"
        },
        "Needs Attention": {
            "description": "Below average on multiple RFM dimensions",
            "action": "Re-engage with targeted offer",
            "campaign": "Category-specific promotion based on past purchases",
            "priority": "Medium"
        },
        "About to sleep": {
            "description": "Declining engagement, at risk of going silent",
            "action": "Wake-up campaign before they go cold",
            "campaign": "Limited time offer with countdown timer",
            "priority": "High"
        },
        "Anomalous": {
            "description": "Unusual behavior — returns, bulk buying, erratic",
            "action": "Manual review and reclassification",
            "campaign": "No automated campaign — flag for human review",
            "priority": "Low"
        }
    }

In [60]:
def generate_anomaly_insights(summary: dict, df: pd.DataFrame) -> str:
    anomaly_df = df[df["is_anomaly"] == 1]

    top_anomalies = anomaly_df.nlargest(5, "anomaly_score")[
        ["CustomerID", "Monetary", "Frequency",
         "AvgItemsPerOrder", "anomaly_score", "anomaly_type"]
    ].to_dict(orient="records")

    prompt = f"""
You are a fraud analyst and customer behavior specialist.
Analyze these anomalous customers detected in the dataset.

Anomaly Type Breakdown:
{json.dumps(summary['anomaly_summary'], indent=2)}

Top 5 Most Anomalous Customers:
{json.dumps(top_anomalies, indent=2)}
Total anomalies: {summary['total_anomalies']} out of {summary['total_customer']} customers

Provide a brief anomaly report that:
1. Explains what each anomaly type means for the business
2. Assesses the risk level of each type
3. Recommends specific actions for each type
4. Highlights any patterns worth investigating further

Write in clear business language, 3-4 paragraphs.
Do NOT explain your thought process. Give only the summary.
"""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=500    )
    return response.choices[0].message.content

In [42]:
def generate_revenue_alerts(df: pd.DataFrame) -> list:
    """Auto-generate priority business alerts"""

    alerts = []

    # Alert 1 — At Risk high value customers
    at_risk = df[df["Segment"] == "At Risk"]
    at_risk_revenue = at_risk["clv_12months"].sum()
    if len(at_risk) > 0:
        alerts.append({
            "priority": "Critical",
            "type": "Revenue at Risk",
            "title": f"{len(at_risk)} At-Risk Customers",
            "metric": f"£{at_risk_revenue:,.0f} CLV at risk",
            "action": "Launch immediate win-back campaign"
        })

    # Alert 2 — Champions with declining prob_alive
    champions = df[df["Segment"] == "Champion"]
    champ_low_alive = champions[champions["prob_alive"] < 0.5]
    if len(champ_low_alive) > 0:
        alerts.append({
            "priority": "Critical",
            "type": "Champion Churn Risk",
            "title": f"{len(champ_low_alive)} Champions showing churn signals",
            "metric": f"£{champ_low_alive['clv_12months'].sum():,.0f} CLV at risk",
            "action": "Personal outreach and exclusive loyalty offer"
        })

    # Alert 3 — Promising customers
    promising = df[df["Segment"] == "Promising"]
    if len(promising) > 0:
        alerts.append({
            "priority": "High",
            "type": "Growth Opportunity",
            "title": f"{len(promising)} Promising customers ready to convert",
            "metric": f"Avg first order £{promising['AvgOrderValue'].mean():,.0f}",
            "action": "Fast-track loyalty onboarding within 7 days"
        })

    # Alert 4 — Bulk buyers
    bulk = df[df["anomaly_type"] == "Bulk Buyer / Reseller"]
    if len(bulk) > 0:
        alerts.append({
            "priority": "High",
            "type": "Wholesale Opportunity",
            "title": f"{len(bulk)} potential wholesale/reseller accounts",
            "metric": f"Combined spend £{bulk['Monetary'].sum():,.0f}",
            "action": "Move to B2B track with dedicated account manager"
        })

    # Alert 5 — Can't lose them
    cant_lose = df[df["Segment"] == "Can't Lose Them"] if "Can't Lose Them" in df["Segment"].values else pd.DataFrame()
    if len(cant_lose) > 0:
        alerts.append({
            "priority": "Critical",
            "type": "High Value Churn",
            "title": f"{len(cant_lose)} high-value customers going silent",
            "metric": f"£{cant_lose['clv_12months'].sum():,.0f} CLV at risk",
            "action": "Immediate personal outreach required"
        })

    return sorted(alerts, key=lambda x:
        {"Critical": 0, "High": 1, "Medium": 2, "Low": 3}[x["priority"]]
    )

In [65]:
def answer_business_question(question: str, df: pd.DataFrame, summary: dict) -> str:
    """Natural language Q&A about the customer data"""

    # Build context from data
    context = f"""
    You have access to customer analytics data with these key facts:

    Total customers: {summary['total_customer']}
    Total revenue: £{summary['total_revenue']:,.2f}

    Segments and sizes:
    {json.dumps(summary['segment_summary']['count'], indent=2)}

    Average monetary per segment:
    {json.dumps(summary['segment_summary']['avg_monetary'], indent=2)}

    Average CLV per segment:
    {json.dumps(summary['segment_summary']['avg_clv'], indent=2)}

    Anomalies detected:
    {json.dumps(summary['anomaly_summary'], indent=2)}
    """
    prompt = f"""
You are a customer analytics AI assistant.
Answer the following business question using the data provided.
Be specific, cite numbers, and give actionable advice.

Data Context:
{context}

Question: {question}

Answer in 2-3 paragraphs maximum. Be direct and specific.
Do NOT explain your thought process. Give only the summary.
"""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=400
    )
    return response.choices[0].message.content

In [62]:
print("=" * 60)
print("AUTOMATED CUSTOMER INTELLIGENCE REPORT")
print("=" * 60)

print("\n EXECUTIVE SUMMARY")
print("-" * 40)
executive_summary = generate_executive_summary(summary)
print(executive_summary)

AUTOMATED CUSTOMER INTELLIGENCE REPORT

 EXECUTIVE SUMMARY
----------------------------------------
<think>
Okay, let's tackle this. The user wants an executive summary based on customer data. First, I need to highlight the most important findings. The total customers are 4,372 with a total revenue of over £8 million. The average predicted CLV is around £3,096, which is a key metric. Then there are 219 anomalous customers, but only 15 are classified under the "Anomalous" segment. That seems low, maybe the rest are spread out? Wait, the dataset mentions 219 anomalous detected, but in the segment distribution, "Anomalous" is 15. Maybe the rest are anomalies in other segments? Hmm, need to clarify that in the summary.

Next, the segment distribution. The largest segments are "Champion" with 972 and "Lost Customers" with 809. The monetary value for "Champion" is the highest at £5,804.84, which is a big contributor to revenue. The CLV for "Champion" is also the highest at £5,421.30. That's 

In [47]:
print("\n SEGMENT RECOMMENDATIONS")
print("-" * 40)
recommendations = generate_segment_recommendations(summary)
for segment, rec in recommendations.items():
    print(f"\n{segment} [{rec['priority']}]")
    print(f"  Who:      {rec['description']}")
    print(f"  Action:   {rec['action']}")
    print(f"  Campaign: {rec['campaign']}")


 SEGMENT RECOMMENDATIONS
----------------------------------------
Raw model response:
<think>
Okay, let's tackle this. The user wants me to generate specific, actionable recommendations for each customer segment based on their RFM data. First, I need to understand each segment's characteristics from the provided averages.

Starting with "About to sleep": They have a recency of 52 days, which is pretty high. Frequency is low (1.15), and monetary is moderate. Probability of being active is 0.91. So they're not lost yet but might be. The key action here is probably to re-engage them before they become inactive. Maybe a personalized offer to encourage a repeat purchase.

"Anomalous" has a higher recency (89 days) but higher frequency (2.07) and lower monetary (64.1). Probability is 0.82. They might be inconsistent buyers. The action could be to investigate why they're not buying as expected. Maybe a survey or a special offer to bring them back.

"At Risk" has a recency of 137 days, freque

In [50]:
print("\nANOMALY INSIGHTS")
print("-" * 40)
anomaly_insights = generate_anomaly_insights(summary, df)
print(anomaly_insights)


ANOMALY INSIGHTS
----------------------------------------
<think>
Okay, let's tackle this query. The user is a fraud analyst and customer behavior specialist who needs an anomaly report based on the provided data. First, I need to understand the different anomaly types and their implications.

Starting with the breakdown: Erratic Behavior is the most common with 125 cases, followed by Bulk Buyers, then Ghost Customers and Return Abuses. The top 5 customers are all Bulk Buyers except one Erratic Behavior case. 

For each anomaly type, I need to explain what they mean for the business. Erratic Behavior likely refers to unpredictable purchasing patterns, which might indicate fraud or testing. Bulk Buyers are resellers, which could mean they're not the end-users, affecting profit margins. Ghost Customers might be fake accounts, and Return Abuses are obvious in their risk of losses from returns.

Next, assessing risk levels. Ghost Customers and Return Abuses are high risk because they dire

In [63]:
print("\nPRIORITY ALERTS")
print("-" * 40)
alerts = generate_revenue_alerts(df)
for alert in alerts:
    print(f"\n[{alert['priority']}] {alert['title']}")
    print(f"  Metric: {alert['metric']}")
    print(f"  Action: {alert['action']}")


PRIORITY ALERTS
----------------------------------------

[Critical] 449 At-Risk Customers
  Metric: £703,155 CLV at risk
  Action: Launch immediate win-back campaign

[Critical] 1 Champions showing churn signals
  Metric: £201 CLV at risk
  Action: Personal outreach and exclusive loyalty offer

[High] 22 Promising customers ready to convert
  Metric: Avg first order £210
  Action: Fast-track loyalty onboarding within 7 days

[High] 85 potential wholesale/reseller accounts
  Metric: Combined spend £1,852,003
  Action: Move to B2B track with dedicated account manager


In [66]:
print("\nNATURAL LANGUAGE Q&A TEST")
print("-" * 40)
test_questions = [
    "Which customers should I prioritize for Black Friday?",
    "What is my biggest revenue risk right now?",
    "Which segment has the best growth potential?"
]

for q in test_questions:
    print(f"\nQ: {q}")
    print(f"A: {answer_business_question(q, df, summary)}")


NATURAL LANGUAGE Q&A TEST
----------------------------------------

Q: Which customers should I prioritize for Black Friday?
A: <think>
Okay, let's tackle this. The user wants to know which customers to prioritize for Black Friday. They provided customer segments with sizes, average monetary value, and CLV.

First, I need to identify which segments contribute the most to revenue and have high CLV. Champions have the highest average monetary value (£5804.84) and CLV (£5421.3), so they're a top priority. They're 972 customers, which is a decent size. Next, Loyal Customers have a high CLV (£2017.76) and monetary value (£1706.86), so they should be next. 

Potential Loyalists have a good CLV (£2085.01) and monetary value (£1052.68), so they might be worth targeting too. At Risk customers have a high monetary value (£1519.87) but lower CLV. Maybe they need retention strategies. 

Anomalies like Erratic Behavior and Bulk Buyers might not be the focus unless there's a strategy to convert the

In [67]:
insights_output = {
    "executive_summary":    executive_summary,
    "segment_recommendations": recommendations,
    "anomaly_insights":     anomaly_insights,
    "alerts":               alerts,
}

with open("insights.json", "w") as f:
    json.dump(insights_output, f, indent=2)

print("\nSaved: insights.json")


Saved: insights.json
